# Local Projection: geopolitical risk shock

## Geopolitical risk index
### The GPR index
The Geopolitical Risk Index (GPR) from Measuring Geopolitical Risk (AER, 2018) by Matteo Iacoviello and Dario Caldara is a quantitative measure of global geopolitical tensions.

The GPR captures the intensity of geopolitical risk by counting how often major international newspapers mention terms related to geopolitical tensions—such as wars, terrorism, and international conflicts. The index is scaled over time to show periods of higher or lower risk.

The GPR index used is monthly. To be used, it is aggregated to annual frequency. 

### The GPR Shock
The geopolitical shock is identified as the innovation/unexpected component of GPR series for each country.

The geopolitical risk shock is identified as the unexpected component of the country-specific GPR series after removing its predictable dynamics. First, the panel is structured by country and year, and for each country separately, an autoregressive model of order one (AR(1)) is estimated, where the current value of the geopolitical risk index is regressed on its lagged value. This step captures the persistence in geopolitical risk. The shock is then defined as the regression residual—i.e., the portion of geopolitical risk that cannot be explained by its own past. By construction, this residual represents an innovation or surprise change in geopolitical risk, isolating exogenous fluctuations from predictable trends.

In [2]:
use "/Users/2941628y/Library/CloudStorage/OneDrive-UniversityofGlasgow/Desktop/LP/gpr_annual.dta", clear

*----- shock identification -----*
encode fic, gen(fic_code)


** AR(1) model **
xtset fic_code fyear

gen gpr_shock = .


levelsof fic, local(countries)

foreach c of local countries {
    reg gprc L.gprc if fic == "`c'"
    predict temp_resid if fic == "`c'", resid
    replace gpr_shock = temp_resid if fic == "`c'"
    drop temp_resid
}

sum gprc gpr_shock





Panel variable: fic_code (strongly balanced)
 Time variable: fyear, 1900 to 2026
         Delta: 1 unit

(5,588 missing values generated)

`"ARG"' `"AUS"' `"BEL"' `"BRA"' `"CAN"' `"CHE"' `"CHL"' `"CHN"' `"COL"' `"DEU"' 
> `"DNK"' `"EGY"' `"ESP"' `"FIN"' `"FRA"' `"GBR"' `"HKG"' `"HUN"' `"IDN"' `"IND"
> ' `"ISR"' `"ITA"' `"JPN"' `"KOR"' `"MEX"' `"MYS"' `"NLD"' `"NOR"' `"PER"' `"PH
> L"' `"POL"' `"PRT"' `"RUS"' `"SAU"' `"SWE"' `"THA"' `"TUN"' `"TUR"' `"TWN"' `"
> UKR"' `"USA"' `"VEN"' `"VNM"' `"ZAF"'


      Source |       SS           df       MS      Number of obs   =        41
-------------+----------------------------------   F(1, 39)        =      2.69
       Model |   .00096868         1   .00096868   Prob > F        =    0.1093
    Residual |  .014069875        39  .000360766   R-squared       =    0.0644
-------------+----------------------------------   Adj R-squared   =    0.0404
       Total |  .015038554        40  .000375964   Root MSE        =    .01899

----------------

## Local projection: geopolitical shock
The empirical specification estimates the dynamic impact of geopolitical risk shocks on firm-level employment growth using a local projection framework. 

The employment growth is constructed as a symmetric growth rate between consecutive periods to avoid scale distortions following the  Davis, Haltiwanger, and Schuh (1996)(thereafter DHS) employment growth rate calculation. 

Firm size is controlled for using the logarithm of employment and its lag, capturing persistence in firm dynamics. 

The key explanatory variable is the lagged geopolitical risk shock, which ensures a clear temporal ordering between the shock and firm outcomes. 

The local projection method is then applied to trace the response of employment growth over horizons 0 to 5 years following the shock. The specification includes firm size, year fixed effects, industry fixed effects, and country fixed effects to control for common macroeconomic conditions and structural heterogeneity. 

Estimation is conducted using fixed effects panel regressions, with standard errors clustered at the firm and industry levels to account for within-group correlation. The cumulative transformation of coefficients allows the results to be interpreted as the total effect of a geopolitical risk shock on employment growth over time.

In [4]:
use "/Users/2941628y/Library/CloudStorage/OneDrive-UniversityofGlasgow/Desktop/LP/local_projection.dta", clear

encode fic, gen(fic_code)

** GPR shock data ** 
merge m:1 fic fyear using "/Users/2941628y/Library/CloudStorage/OneDrive-UniversityofGlasgow/Desktop/LP/gpr_shock.dta"
drop if missing(gvkey)
drop _merge



*---- LP setup -----*
xtset gvkey fyear
sort gvkey fyear

** dependent variable 
by gvkey: gen empgrowth = (emp[_n+1] - emp[_n])/(0.5 * emp[_n+1] + 0.5 * emp[_n])

** firm control 
gen logemp = log(emp)
gen lag_size = L.logemp

** country code
encode fic, gen(fic_num)

** shock variable
gen lgpr_shock = L.gpr_shock



** local projection **
locproj empgrowth, shock(lgpr_shock) hor(0/5) controls(lag_size i.fyear i.sic i.fic_num) yl(1) sl(1) transf(cmlt) met(xtreg) fe vce(cluster gvkey sic) fact(100) zero




(label fic_code already defined)

    Result                      Number of obs
    -----------------------------------------
    Not matched                        25,727
        from master                    21,482  (_merge==1)
        from using                      4,245  (_merge==2)

    Matched                           171,353  (_merge==3)
    -----------------------------------------

(4,245 observations deleted)



Panel variable: gvkey (unbalanced)
 Time variable: fyear, 1986 to 2024, but with gaps
         Delta: 1 unit


(98,847 missing values generated)

(83,342 missing values generated)

(100,530 missing values generated)


(49,819 missing values generated)


Impulse Response Function

             |       IRF   Std.Err.    IRF LOW     IRF UP 
-------------+-------------------------------------------
           0 |   0.48760    0.36871   -0.23717    1.21236 
           1 |  -0.65638    0.49577   -1.63095    0.31820 
           2 |  -1.83721    0.64961   -3.11424   -0.

## Local Projection: by country
country classification based on: income type(high-income country, low-income country, etc); region(North America, European and Pacific, etc.); and by development(developed country, developing country etc)

In [ ]:
*----- By Region -----*
** World Bank Regional Classification **
gen region_wb = ""

* North America (BENCHMARK)
replace region_wb = "NAM" if inlist(fic, "USA", "CAN")

* Europe & Central Asia
replace region_wb = "ECA" if inlist(fic, "AUT","BEL","CHE","CYP","CZE","DEU","DNK") 
replace region_wb = "ECA" if inlist(fic, "ESP","EST","FIN","FRA","GBR","GRC","HRV")
replace region_wb = "ECA" if inlist(fic, "HUN","IRL","ISL","ITA","LIE","LTU","LUX")
replace region_wb = "ECA" if inlist(fic, "LVA","MCO","MLT","NLD","NOR","POL","PRT")
replace region_wb = "ECA" if inlist(fic, "ROU","RUS","SRB","SVK","SVN","SWE","TUR")
replace region_wb = "ECA" if inlist(fic, "BGR","UKR","MKD","KAZ")

* East Asia & Pacific
replace region_wb = "EAP" if inlist(fic, "AUS","CHN","HKG","IDN","JPN","KOR","MYS")
replace region_wb = "EAP" if inlist(fic, "NZL","PHL","SGP","THA","TWN","VNM","PNG")

* South Asia
replace region_wb = "SAS" if inlist(fic, "BGD","IND","LKA","PAK")

* Latin America & Caribbean
replace region_wb = "LAC" if inlist(fic, "ARG","BRA","CHL","COL","JAM","MEX","PAN")
replace region_wb = "LAC" if inlist(fic, "PER","TTO","VEN","CYM","BMU")

* Middle East & North Africa
replace region_wb = "MNA" if inlist(fic, "ARE","BHR","EGY","ISR","JOR","KWT","MAR")
replace region_wb = "MNA" if inlist(fic, "OMN","QAT","SAU","TUN","EMU")

* Sub-Saharan Africa
replace region_wb = "SSA" if inlist(fic, "BFA","BWA","CIV","GHA","KEN","MUS","MWI")
replace region_wb = "SSA" if inlist(fic, "NAM","NGA","SEN","TZA","UGA","ZAF","ZMB")
replace region_wb = "SSA" if inlist(fic, "ZWE")

* summary
tab region_wb


* shock interaction
gen reg_eca = region_wb == "ECA"
gen reg_eap = region_wb == "EAP"
gen reg_sas = region_wb == "SAS"
gen reg_lac = region_wb == "LAC"
gen reg_mna = region_wb == "MNA"
gen reg_ssa = region_wb == "SSA"
gen reg_nam = region_wb == "NAM"

gen shock_eca = lgpr_shock * reg_eca
gen shock_eap = lgpr_shock * reg_eap
gen shock_sas = lgpr_shock * reg_sas
gen shock_lac = lgpr_shock * reg_lac
gen shock_mna = lgpr_shock * reg_mna
gen shock_ssa = lgpr_shock * reg_ssa
gen shock_nam = lgpr_shock * reg_na

* LP
xtset gvkey fyear

locproj empgrowth, ///
    shock(shock_nam shock_eca shock_eap shock_sas shock_lac shock_mna shock_ssa) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_nam) ///
	nograph save irfname("na_irf") 

locproj empgrowth, ///
    shock(shock_nam shock_eca shock_eap shock_sas shock_lac shock_mna shock_ssa) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_nam + shock_eca) ///
	nograph save irfname("euro_irf")

lpgraph na_irf euro_irf, hor(0/5) zero 

locproj empgrowth, ///
    shock(shock_nam shock_eca shock_eap shock_sas shock_lac shock_mna shock_ssa) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_nam + shock_sas)
	
locproj empgrowth, ///
    shock(shock_nam shock_eca shock_eap shock_sas shock_lac shock_mna shock_ssa) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
	lcs(shock_nam + shock_eap)
	
locproj empgrowth, ///
    shock(shock_nam shock_eca shock_eap shock_sas shock_lac shock_mna shock_ssa) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
	lcs(shock_nam + shock_lac)
	
locproj empgrowth, ///
    shock(shock_nam shock_eca shock_eap shock_sas shock_lac shock_mna shock_ssa) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
	lcs(shock_nam + shock_mna)
	
locproj empgrowth, ///
    shock(shock_nam shock_eca shock_eap shock_sas shock_lac shock_mna shock_ssa) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
	lcs(shock_nam + shock_ssa)


	
** Broad region **
gen region_broad = "OTH"

* North America
replace region_broad = "NAM" if inlist(fic, "USA", "CAN")

* Europe only (exclude Central Asia)
replace region_broad = "EUR" if ///
    inlist(fic, "AUT","BEL","CHE","CYP","CZE","DEU","DNK") | ///
    inlist(fic, "ESP","EST","FIN","FRA","GBR","GRC","HRV") | ///
    inlist(fic, "HUN","IRL","ISL","ITA","LIE","LTU","LUX") | ///
    inlist(fic, "LVA","MCO","MLT","NLD","NOR","POL","PRT") | ///
    inlist(fic, "ROU","RUS","SRB","SVK","SVN","SWE","TUR") | ///
    inlist(fic, "BGR","UKR","MKD")

* East Asia only (exclude Pacific and Southeast Asia)
replace region_broad = "EAS" if ///
    inlist(fic, "CHN","HKG","JPN","KOR","MNG","TWN")

* Africa as a whole
replace region_broad = "AFR" if ///
    inlist(fic, "DZA","AGO","BEN","BFA","BDI","CMR","CPV") | ///
    inlist(fic, "CAF","TCD","COM","COD","COG","CIV","DJI") | ///
    inlist(fic, "EGY","GNQ","ERI","SWZ","ETH","GAB","GMB") | ///
    inlist(fic, "GHA","GIN","GNB","KEN","LSO","LBR","LBY") | ///
    inlist(fic, "MDG","MWI","MLI","MRT","MUS","MAR","MOZ") | ///
    inlist(fic, "NAM","NER","NGA","RWA","STP","SEN","SYC") | ///
    inlist(fic, "SLE","SOM","ZAF","SSD","SDN","TZA","TGO") | ///
    inlist(fic, "TUN","UGA","ZMB","ZWE")

* Summary
tab region_broad

* shock interaction
gen reg_eur = region_broad == "EUR"
gen reg_eas = region_broad == "EAS"
gen reg_afr = region_broad == "AFR"
gen reg_oth = region_broad == "OTH"
gen reg_NA = region_broad == "NAM"


gen shock_NA = lgpr_shock * reg_NA
gen shock_eur = lgpr_shock * reg_eur
gen shock_eas = lgpr_shock * reg_eas
gen shock_afr = lgpr_shock * reg_afr
gen shock_oth = lgpr_shock * reg_oth

*LP
* North America
locproj empgrowth, ///
    shock(shock_NA shock_eur shock_eas shock_afr) ///
    controls(lag_size i.fyear i.sic i.fic_num shock_oth) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA) ///
    nograph save irfname("na_irf")

* Europe
locproj empgrowth, ///
    shock(shock_NA shock_eur shock_eas shock_afr) ///
    controls(lag_size i.fyear i.sic i.fic_num shock_oth) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA + shock_eur) ///
    nograph save irfname("eur_irf")

* East Asia
locproj empgrowth, ///
    shock(shock_NA shock_eur shock_eas shock_afr) ///
    controls(lag_size i.fyear i.sic i.fic_num shock_oth) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA + shock_eas) ///
    nograph save irfname("eas_irf")

* Africa
locproj empgrowth, ///
    shock(shock_NA shock_eur shock_eas shock_afr) ///
    controls(lag_size i.fyear i.sic i.fic_num shock_oth) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA + shock_afr) ///
    nograph save irfname("afr_irf")

* Plot together
lpgraph na_irf eur_irf eas_irf, hor(0/5) zero





*----- By Income Type -----*
** country type by income **
merge m:1 fic fyear using "/Users/2941628y/Library/CloudStorage/OneDrive-UniversityofGlasgow/Desktop/LP/Data_Source/Country_Type_ByIncome/WBG_Type_Income.dta"
drop if missing(gvkey)
drop Country _merge 
drop Type_cate

** high-income countries: benchmark
gen type_h = .
replace type_h = 1 if Type == "H"
replace type_h = 0 if Type == "L"| Type == "LM"| Type == "UM"

** low-income countries
gen type_l = .
replace type_l = 1 if Type == "L"
replace type_l = 0 if Type == "H" | Type == "LM" | Type == "UM"

** Upper-Middle income
gen type_um =.
replace type_um = 1 if Type == "UM"
replace type_um = 0 if Type == "L" | Type == "H" | Type == "LM"

** Lower-Middle Income
gen type_lm = .
replace type_lm = 1 if Type == "LM"
replace type_lm = 0 if Type == "UM"| Type == "L"| Type == "H"

** shock interaction 
gen shock_H  = lgpr_shock
gen shock_L  = lgpr_shock * type_l
gen shock_UM = lgpr_shock * type_um
gen shock_LM = lgpr_shock * type_lm



** LP
* High income
locproj empgrowth, ///
    shock(shock_H shock_L shock_UM shock_LM) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_H) ///
    nograph save irfname("high_irf")

* Low income
locproj empgrowth, ///
    shock(shock_H shock_L shock_UM shock_LM) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_H + shock_L) ///
    nograph save irfname("low_irf")

* Upper-middle income
locproj empgrowth, ///
    shock(shock_H shock_L shock_UM shock_LM) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_H + shock_UM) ///
    nograph save irfname("um_irf")

* Lower-middle income
locproj empgrowth, ///
    shock(shock_H shock_L shock_UM shock_LM) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_H + shock_LM) ///
    nograph save irfname("lm_irf")
	
* graph
lpgraph high_irf low_irf um_irf lm_irf, hor(0/5) zero





*----- By development -----*
gen region_dm = ""

* Developed Markets (BENCHMARK = North America kept separate)
replace region_dm = "NAM" if inlist(fic, "USA", "CAN")         // Benchmark

replace region_dm = "DM"  if inlist(fic, "AUS","AUT","BEL","CHE","DEU","DNK")
replace region_dm = "DM"  if inlist(fic, "ESP","FIN","FRA","GBR","GRC","HKG")
replace region_dm = "DM"  if inlist(fic, "IRL","ISL","ISR","ITA","JPN","KOR")
replace region_dm = "DM"  if inlist(fic, "LUX","NLD","NOR","NZL","PRT","SGP")
replace region_dm = "DM"  if inlist(fic, "SWE","TWN","LIE","MCO","CYM","BMU")

* Emerging Markets — Asia
replace region_dm = "EM_ASIA" if inlist(fic, "BGD","CHN","IDN","IND","MYS")
replace region_dm = "EM_ASIA" if inlist(fic, "PAK","PHL","THA","VNM","LKA","PNG")

* Emerging Markets — Europe & Central Asia
replace region_dm = "EM_ECA"  if inlist(fic, "BGR","CZE","EST","HRV","HUN","KAZ")
replace region_dm = "EM_ECA"  if inlist(fic, "LTU","LVA","MKD","POL","ROU","RUS")
replace region_dm = "EM_ECA"  if inlist(fic, "SRB","SVK","SVN","TUR","UKR")

* Emerging Markets — Latin America
replace region_dm = "EM_LAC"  if inlist(fic, "ARG","BRA","CHL","COL","JAM","MEX")
replace region_dm = "EM_LAC"  if inlist(fic, "PAN","PER","TTO","VEN")

* Emerging Markets — Middle East & Africa
replace region_dm = "EM_MEA"  if inlist(fic, "ARE","BHR","BWA","CIV","EGY","GHA")
replace region_dm = "EM_MEA"  if inlist(fic, "JOR","KEN","KWT","MAR","MUS","MWI")
replace region_dm = "EM_MEA"  if inlist(fic, "NAM","NGA","OMN","QAT","SAU","SEN")
replace region_dm = "EM_MEA"  if inlist(fic, "TUN","TZA","UGA","ZAF","ZMB","ZWE")
replace region_dm = "EM_MEA"  if inlist(fic, "BFA","CYP","EMU","IMN","MLT")

* Check for missing
tab region_dm

* shock interaction
gen reg_dm      = region_dm == "DM"
gen reg_em_asia = region_dm == "EM_ASIA"
gen reg_em_eca  = region_dm == "EM_ECA"
gen reg_em_lac  = region_dm == "EM_LAC"
gen reg_em_mea  = region_dm == "EM_MEA"
gen reg_na      = region_dm == "NAM"

gen shock_NA      = lgpr_shock * reg_na
gen shock_dm      = lgpr_shock * reg_dm
gen shock_em_asia = lgpr_shock * reg_em_asia
gen shock_em_eca  = lgpr_shock * reg_em_eca
gen shock_em_lac  = lgpr_shock * reg_em_lac
gen shock_em_mea  = lgpr_shock * reg_em_mea

* North America (benchmark)
locproj empgrowth, ///
    shock(shock_NA shock_dm shock_em_asia shock_em_eca shock_em_lac shock_em_mea) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA) ///
    nograph save irfname("na_irf")

* Developed Markets
locproj empgrowth, ///
    shock(shock_NA shock_dm shock_em_asia shock_em_eca shock_em_lac shock_em_mea) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA + shock_dm) ///
    nograph save irfname("dm_irf")

* Emerging Markets - Asia
locproj empgrowth, ///
    shock(shock_NA shock_dm shock_em_asia shock_em_eca shock_em_lac shock_em_mea) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA + shock_em_asia) ///
    nograph save irfname("em_asia_irf")

* Emerging Markets - Europe & Central Asia
locproj empgrowth, ///
    shock(shock_NA shock_dm shock_em_asia shock_em_eca shock_em_lac shock_em_mea) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA + shock_em_eca) ///
    nograph save irfname("em_eca_irf")

* Emerging Markets - Latin America
locproj empgrowth, ///
    shock(shock_NA shock_dm shock_em_asia shock_em_eca shock_em_lac shock_em_mea) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA + shock_em_lac) ///
    nograph save irfname("em_lac_irf")

* Emerging Markets - Middle East & Africa
locproj empgrowth, ///
    shock(shock_NA shock_dm shock_em_asia shock_em_eca shock_em_lac shock_em_mea) ///
    controls(lag_size i.fyear i.sic i.fic_num) ///
    yl(1) sl(1) transf(cmlt) met(xtreg) fe ///
    vce(cluster gvkey sic) fact(100) zero ///
    lcs(shock_NA + shock_em_mea) ///
    nograph save irfname("em_mea_irf")

lpgraph na_irf dm_irf em_asia_irf em_eca_irf, hor(0/5) zero